In [27]:
!pip install -q gradio langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu transformers pypdf torch

In [28]:
import gradio as gr
import os
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [29]:
# ---------------------------------------------------------
# 3. GLOBAL VARIABLES & MODEL LOADING
# ---------------------------------------------------------
print("Step 2: Loading AI models...")
# Using the same logic from your rag.py bootcamp script
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Global variable to hold our document index
vector_db = None

Step 2: Loading AI models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [30]:
# ---------------------------------------------------------
# 4. CORE LOGIC FUNCTIONS
# ---------------------------------------------------------
def process_pdf(file):
    global vector_db
    if file is None:
        return "❌ Error: Please upload a PDF first."

    try:
        # Load PDF using PyPDFLoader
        loader = PyPDFLoader(file.name)
        documents = loader.load()

        # Chunking (Same settings as your rag.py)
        splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
        chunks = splitter.split_documents(documents)

        # Embedding & Storage (FAISS)
        vector_db = FAISS.from_documents(chunks, embeddings)
        return f"✅ Success! PDF split into {len(chunks)} chunks and indexed."
    except Exception as e:
        return f"❌ Error processing PDF: {str(e)}"

def respond(message, history):
    global vector_db
    if vector_db is None:
        return "Please upload and process a PDF in the left panel first!"

    # Similarity Search
    results = vector_db.similarity_search(message, k=3)
    context = "\n".join([r.page_content for r in results])

    # Prompt Construction
    prompt = f"Answer the question using the context below.\n\nContext:\n{context}\n\nQuestion:\n{message}\n\nAnswer:"

    # Generate Response
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=150)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [31]:
# ---------------------------------------------------------
# 5. GRADIO USER INTERFACE
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 RAG Study Buddy\n*Built based on Bootcamp logic*")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🛠 Setup")
            file_input = gr.File(label="Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("Process Document", variant="primary")
            status = gr.Textbox(label="System Status", placeholder="Awaiting upload...", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### 💬 Chat")
            chat_ui = gr.ChatInterface(
                fn=respond,
                examples=["What is the main topic?", "Summarize the document"],
                cache_examples=False,
            )

    # Link the button to the processing function
    process_btn.click(fn=process_pdf, inputs=file_input, outputs=status)

# Launching with 'inline=True' for Colab display and 'share=True' for a public link
demo.launch(inline=True, share=True, debug=True)

/tmp/ipykernel_275/912303178.py:4: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://34b30a490ed19d5948.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://34b30a490ed19d5948.gradio.live
